# SuppFig — bulk-coverage (BULKCOV) distribution per scNanoSeq sample

Per-sample kernel-density estimate of the matched bulk WGS coverage (BULKCOV) at each scNanoSeq variant site (sites with BULKCOV ≤ 200). The solid line marks the BULKCOV = 20 cutoff used downstream; the dashed line is the per-sample median. Panel titles report the fraction of sites below 20×.

Input (precomputed, in `data/SuppFig_BULKCOV_distribution/`): `bulkcov_values.csv` — long-form `Sample`, `BULKCOV` (≤ 200).

In [ ]:
# --- Nature figure style: 0.25 lw spines/ticks, no top/right spines, embedded fonts ---
import matplotlib as mpl
mpl.rcParams.update({
    "pdf.fonttype": 42, "ps.fonttype": 42, "font.family": "sans-serif", "font.size": 8,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.linewidth": 0.25,
    "xtick.major.width": 0.25, "ytick.major.width": 0.25,
    "xtick.minor.width": 0.25, "ytick.minor.width": 0.25,
    "xtick.labelsize": 7, "ytick.labelsize": 7,
})

def nature_axes(ax, keep_top=False):
    ax.spines["right"].set_visible(False)
    ax.spines["top"].set_visible(keep_top)
    for sp in ax.spines.values():
        sp.set_linewidth(0.25)
    ax.tick_params(width=0.25)
    return ax

import os
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

OUT = "../output/SuppFig_BULKCOV_distribution"
os.makedirs(OUT, exist_ok=True)

df = pd.read_csv("data/SuppFig_BULKCOV_distribution/bulkcov_values.csv")
bulkcov = {s: g["BULKCOV"].values for s, g in df.groupby("Sample")}
print(f"samples: {len(bulkcov)} | sites: {len(df)}")

In [ ]:
samples = sorted(bulkcov)
ncol = 5
nrow = (len(samples) + ncol - 1) // ncol
fig, axes = plt.subplots(nrow, ncol, figsize=(ncol * 2.4, nrow * 2.0), squeeze=False)

for idx, s in enumerate(samples):
    ax = axes[idx // ncol][idx % ncol]
    v = bulkcov[s]
    sns.kdeplot(v, ax=ax, fill=True, color="lightgrey", linewidth=0.5)
    ax.axvline(20, color="black", linestyle="-", linewidth=0.5, label="BULKCOV = 20")
    ax.axvline(np.median(v), color="red", linestyle="--", linewidth=0.5, label="Median")
    ax.set_xlim(0, 200)
    frac_lt20 = np.mean(v < 20) * 100
    ax.set_title(f"{s} ({frac_lt20:.1f}% < 20)", fontsize=7)
    ax.set_xlabel("BULKCOV"); ax.set_ylabel("Density")
    nature_axes(ax)
    if idx == 0:
        ax.legend(fontsize=5, frameon=False)

for idx in range(len(samples), nrow * ncol):
    axes[idx // ncol][idx % ncol].axis("off")

fig.tight_layout()
fig.savefig(f"{OUT}/BULKCOV_distribution.pdf", format="pdf", bbox_inches="tight")
plt.show()